In [ ]:
%pip install -q bert_score
%pip install -q ragas
%pip install -q pypdf

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
questions = [
    "아진산업의 최초 설립일은 언제인가요?",
    "아진산업의 2025년 매출액은 얼마인가요?",
    "아진산업의 본사 위치는 어디인가요?",
]

ground_truths = [
    "아진산업은 1978년 5월 31일 창립하였습니다.",
    "아진산업은 2025년에 1조 88억 6783만원의 매출 실적을 달성했습니다.",
    "아진산업의 본사는 경상북도 경산시 진량읍에 있습니다."
]

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("..\\docs\\knu.pdf")
documents = loader.load()

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
llm = ChatOpenAI(model="gpt-5.4-nano")
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding
)
retriever = vectorstore.as_retriever()

In [ ]:
def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
    다음 문맥만 사용하여 질문에 한 문장으로 답하세요.
    문맥에서 알 수 없는 내용은 모른다고 답하세요.
    [문맥] {context}
    [질문] {question}
""")

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
contexts = []
answers = []
for question in questions:
    docs = retriever.invoke(question)
    contexts.append([doc.page_content for doc in docs])
    answer = rag_chain.invoke(question)
    answers.append(answer)

In [ ]:
import pandas as pd
from bert_score import score

P, R, F1 = score(
    cands=answers,
    refs=ground_truths,
    lang="ko",
    verbose=True
)

df_bertscore = pd.DataFrame({
    "question": questions,
    "ground_truth": ground_truths,
    "answer": answers,
    "bertscore_precision": P.tolist(),
    "bertscore_recall": R.tolist(),
    "bertscore_f1": F1.tolist(),
})

df_bertscore

In [ ]:
df_bertscore["bertscore_f1"].mean()

In [ ]:
from datasets import Dataset

ragas_data = {
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths,
}

dataset = Dataset.from_dict(ragas_data)
dataset

In [ ]:
dataset[0]

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness, answer_relevancy,
    context_precision, context_recall,
)

ragas_result = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ],
    llm=llm,
    embeddings=embedding,
)
ragas_result

In [ ]:
df_ragas = ragas_result.to_pandas()
df_ragas

In [ ]:
df_eval = df_ragas.copy()

df_eval["bertscore_precision"] = df_bertscore["bertscore_precision"]
df_eval["bertscore_recall"] = df_bertscore["bertscore_recall"]
df_eval["bertscore_f1"] = df_bertscore["bertscore_f1"]

df_eval[
    [
        "question",
        "ground_truth",
        "answer",
        "bertscore_f1",
        "faithfulness",
        "answer_relevancy",
        "context_precision",
        "context_recall",
    ]
]

In [ ]:
score_columns = [
    "bertscore_f1",
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
]

df_eval[score_columns].mean()

In [ ]:
import sys
print(sys.executable)
print(sys.version)

In [ ]:
import sys
!{sys.executable} -m pip show bert-score
!{sys.executable} -m pip show ragas
!{sys.executable} -m pip show langchain-google-vertexai

In [ ]:
import sys

!{sys.executable} -m pip install bert-score ragas langchain-google-vertexai

In [ ]:
import os
os._exit(00)